In [ ]:
import pandas as pd
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
pd.set_option('display.max_columns', None)

In [ ]:
# Define the base path
base_path = "./data-all-waves"
file_path = os.path.join(base_path, "df_final.csv")
df_final = pd.read_csv(file_path)

In [ ]:
# Function to clean EDMOM and EDDAD values
def clean_ed(x):
    if pd.isna(x):  # Keep NaN as is
        return x
    if isinstance(x, str):
        if x.isdigit():  # Convert string numbers to int
            return int(x)
        elif x == 'E':   # Keep 'E' as is
            return 'E'
    return x  # Return original value if it's already an integer

# Apply the cleaning function to EDMOM and EDDAD for all years
df_final['EDMOM'] = df_final['EDMOM'].apply(clean_ed)
df_final['EDDAD'] = df_final['EDDAD'].apply(clean_ed)

# Verify the cleaned values
for year in sorted(df_final['REFYR'].unique()):
    edmom_values = df_final[df_final['REFYR'] == year]['EDMOM'].unique()
    eddad_values = df_final[df_final['REFYR'] == year]['EDDAD'].unique()
    
    print(f"\nREFYR {year}:")
    print(sorted([x for x in edmom_values if x != 'E']) + (['E'] if 'E' in edmom_values else []))
    print(sorted([x for x in eddad_values if x != 'E']) + (['E'] if 'E' in eddad_values else []))


In [ ]:
import os
import pandas as pd

# Define the path and read the CSV file
base_path = "./data-all-waves"
ur_ba_path = os.path.join(base_path, "ur_ba_16y.csv")

# Read the CSV file
ur_ba_16y = pd.read_csv(ur_ba_path)

# Rename columns
ur_ba_16y = ur_ba_16y.rename(columns={'CGBD16O': 'UR'})

# First prepare yearly unemployment rates for both degree types
# For Bachelor's
ur_ba_16y['observation_date'] = pd.to_datetime(ur_ba_16y['observation_date'])
ur_ba_16y['year'] = ur_ba_16y['observation_date'].dt.year


# Assume there is a 'YEAR' column in ur_ba_16y indicating the year
# If not, check if 'BAYR' corresponds to a year column
if 'year' in ur_ba_16y.columns:
    # Compute the annual average unemployment rate
    ur_avg = ur_ba_16y.groupby('year')['UR'].mean().reset_index()
    
    # Rename YEAR to BAYR so it matches df_final
    ur_avg = ur_avg.rename(columns={'year': 'BAYR'})
else:
    print("Error: No 'year' column found in ur_ba_16y. Please check the dataset structure.")

# Merge with df_final based on BAYR
df_final= df_final.merge(ur_avg, on='BAYR', how='left')

In [ ]:
df_final.columns

In [ ]:
def check_field_codes(df):
    """
    Check if there are any field codes not in the valid range
    """
    valid_codes = [11, 12, 21, 22, 23, 31, 32, 33, 34, 41, 42, 43, 44, 45,
                  51, 52, 53, 54, 55, 56, 57, 61, 62, 63, 64, 71, 72, 73, 
                  74, 75, 76]
    
    # Check NBAMENG
    unique_ba = sorted(df['NBAMENG'].unique())
    invalid_ba = [code for code in unique_ba if code not in valid_codes]
    
    # Check NDGMENG
    unique_ma = sorted(df['NDGMENG'].unique())
    invalid_ma = [code for code in unique_ma if code not in valid_codes]
    
    print("Invalid Bachelor's field codes:", invalid_ba if invalid_ba else "None")
    print("Invalid Master's field codes:", invalid_ma if invalid_ma else "None")
    
    # Print value counts
    print("\nBachelor's field code distribution:")
    print(df['NBAMENG'].value_counts().sort_index())
    print("\nMaster's field code distribution:")
    print(df['NDGMENG'].value_counts().sort_index())

# 使用函数
check_field_codes(df_final)

# Major Group Regression

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from statsmodels.discrete.discrete_model import Probit

def prepare_data_major(df):
    """
    Prepare data for analysis of field switching
    """
    
    df = df[df['BAYR']<2020]
    # 1. Keep only the last observation for each person
    df = df.sort_values('REFYR').drop_duplicates(subset='assigned_id', keep='last')
    
    # 2. Create master's indicators
    df['immediate_masters'] = ((df['DGRDG'] == 2) & 
                             (df['DGRYR'] - df['BAYR'] <= 3))
    df['with_masters'] = (df['DGRDG'] == 2)
    
    # 3. Remove observations with missing major information
    df = df[df['NBAMEMG'].notna() & df['NDGMEMG'].notna()]
    
    # Create cohort trend (centered)
    df['cohort_trend'] = df['BAYR'] - df['BAYR'].min()  # Linear trend
    df['cohort_trend_sq'] = df['cohort_trend']**2       # Quadratic trend
    df['cohort_trend_cub'] = df['cohort_trend']**3       # Quadratic trend
    
    # Create dummy variables for categorical variables
    gender_dummies = pd.get_dummies(df['GENDER'], prefix='gender')
    race_dummies = pd.get_dummies(df['RACEM'], prefix='race')
    
    # 4. Create separate datasets for analysis
    immediate = df[df['immediate_masters']==1].copy()
    all_masters = df[df['with_masters']==1].copy()
    
    # 5. Create field switch indicators
    for dataset in [immediate, all_masters]:
        dataset['field_switch'] = (dataset['NBAMEMG'] != dataset['NDGMEMG']).astype(int)
    
    return immediate, all_masters

def run_probit_analysis_robust_cohort(df, formula):
    """
    Run probit regression using formula interface
    """
    
    model = smf.probit(formula, data=df)
   # Cluster standard errors at the graduation cohort level
    results = model.fit(cov_type='cluster', 
                       cov_kwds={'groups': df['BAYR']})
    
    return results

In [ ]:
# To use:
immediate, all_masters = prepare_data_major(df_final)


formula_basic = '''
field_switch ~ UR + cohort_trend + cohort_trend_sq + AGE + C(GENDER) + C(RACEM)
'''

results_basic_r = run_probit_analysis_robust_cohort(all_masters, formula_basic)
print(results_basic_r.summary())

# By degree type: major group

In [ ]:
def analyze_field_transition_by_degree_type(df):
    """
    Analyze field transition behavior by undergraduate degree type
    Using major group classification NBAMEMG and NDGMEMG
    """
    # First prepare the data using your existing function
    _, all_masters = prepare_data_major(df)
    
    # Define the field names dictionary
    field_names = {
        1: "Computer & Math",
        2: "Bio & Life Sciences",
        3: "Physical Sciences",
        4: "Social Sciences",
        5: "Engineering",
        6: "S&E-Related",
        7: "Non-S&E"
    }
    
    # Create field switch indicator based on major groups
    all_masters['field_switch'] = (all_masters['NBAMEMG'] != all_masters['NDGMEMG']).astype(int)
    
    # Approach 1: Separate regressions for each undergraduate field
    results = {}
    
    # Run separate models for each undergraduate field
    for field_code, field_name in field_names.items():
        # Filter data for this field
        field_df = all_masters[all_masters['NBAMEMG'] == field_code]
        
        # Skip if we don't have enough data
        if len(field_df) < 100:  # Adjust threshold as needed
            print(f"Skipping {field_name} - insufficient data ({len(field_df)} observations)")
            continue
            
        # Run probit model
        formula = 'field_switch ~ UR + cohort_trend + cohort_trend_sq + AGE + C(GENDER) + C(RACEM)'
        results[f'{field_name}_model'] = run_probit_analysis_robust_cohort(field_df, formula)
        
        # Print key results
        print(f"\nResults for {field_name} undergraduate majors:")
        print(results[f'{field_name}_model'].summary())
#         print(f"Coefficient on UR: {results[f'{field_name}_model'].params['UR']:.4f}")
#         print(f"P-value: {results[f'{field_name}_model'].pvalues['UR']:.4f}")
#         print(f"Coefficient on UR: {results[f'{field_name}_model'].params['cohort_trend']:.4f}")
#         print(f"P-value: {results[f'{field_name}_model'].pvalues['cohort_trend']:.4f}")
#         print(f"Coefficient on UR: {results[f'{field_name}_model'].params['cohort_trend_sq']:.4f}")
#         print(f"P-value: {results[f'{field_name}_model'].pvalues['cohort_trend_sq']:.4f}")

analyze_field_transition_by_degree_type(df_final)

# Minor Group fields

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from statsmodels.discrete.discrete_model import Probit

def prepare_data_minor(df):
    """
    Prepare data for analysis of field switching
    """
    
    df = df[df['BAYR']<2020]
    # 1. Keep only the last observation for each person
    df = df.sort_values('REFYR').drop_duplicates(subset='assigned_id', keep='last')
    
    # 2. Create master's indicators
    df['immediate_masters'] = ((df['DGRDG'] == 2) & 
                             (df['DGRYR'] - df['BAYR'] <= 3))
    df['with_masters'] = (df['DGRDG'] == 2)
    
    # 3. Remove observations with missing major information
    df = df[df['NBAMENG'].notna() & df['NDGMENG'].notna()]
    
    # Create cohort trend (centered)
    df['cohort_trend'] = df['BAYR'] - df['BAYR'].min()  # Linear trend
    df['cohort_trend_sq'] = df['cohort_trend']**2       # Quadratic trend
    df['cohort_trend_cub'] = df['cohort_trend']**3       # Quadratic trend
    
    # Create dummy variables for categorical variables
    gender_dummies = pd.get_dummies(df['GENDER'], prefix='gender')
    race_dummies = pd.get_dummies(df['RACEM'], prefix='race')
    
    # 4. Create separate datasets for analysis
    immediate = df[df['immediate_masters']==1].copy()
    all_masters = df[df['with_masters']==1].copy()
    
    # 5. Create field switch indicators
    for dataset in [immediate, all_masters]:
        dataset['field_switch'] = (dataset['NBAMENG'] != dataset['NDGMENG']).astype(int)
    
    return immediate, all_masters

def run_probit_analysis(df, formula):
    """
    Run probit regression using formula interface
    """
    model = smf.probit(formula, data=df)
    results = model.fit(cov_type='HC1')
    
    return results

def run_probit_analysis_robust_cohort(df, formula):
    """
    Run probit regression using formula interface
    """
    
    model = smf.probit(formula, data=df)
   # Cluster standard errors at the graduation cohort level
    results = model.fit(cov_type='cluster', 
                       cov_kwds={'groups': df['BAYR']})
    
    return results

In [ ]:
# To use:
immediate, all_masters = prepare_data_minor(df_final)


formula_basic = '''
field_switch ~ UR
'''

formula_basic = '''
field_switch ~ UR + cohort_trend + cohort_trend_sq
'''
results_basic = run_probit_analysis(all_masters, formula_basic)
print(results_basic.summary())

results_basic_r = run_probit_analysis_robust_cohort(all_masters, formula_basic)
print(results_basic_r.summary())

In [ ]:
def create_time_trends_plot(all_masters):
    """
    Create a time trends plot showing field switching rate and unemployment rate
    over time, with a fitted trend line.
    
    Parameters:
    -----------
    all_masters : pandas DataFrame
        DataFrame for all masters with required variables
    
    Returns:
    --------
    fig : matplotlib figure object
    """
    import matplotlib.pyplot as plt
    import statsmodels.api as sm
    import numpy as np
    import pandas as pd
    
    # Create centered graduation year 
    all_masters['grad_year_centered'] = all_masters['BAYR'] - all_masters['BAYR'].mean()
    all_masters['grad_year_squared'] = all_masters['grad_year_centered']**2
    
    # Calculate cohort-level means
    cohort_data = all_masters.groupby('BAYR').agg({
        'field_switch': 'mean',
        'UR': 'first',
        'grad_year_centered': 'first',
        'grad_year_squared': 'first'
    }).reset_index()
    
    # Create the figure and axis
    fig = plt.figure(figsize=(10, 6))
    ax = fig.add_subplot(111)
    
    # Plot field switching trend
    ax.plot(cohort_data['BAYR'], cohort_data['field_switch'], 'o-', color='blue', markersize=8)
    ax.set_xlabel('Bachelor\'s Graduation Year', fontsize=12)
    ax.set_ylabel('Field Switching Rate', color='blue', fontsize=12)
    ax.tick_params(axis='y', labelcolor='blue')
    ax.set_title('Time Trends in Field Switching and Unemployment', fontsize=14)
    ax.grid(True, linestyle='--', alpha=0.7)
    
    # Set x-axis ticks to integers
    years = cohort_data['BAYR'].unique()
    ax.set_xticks(years)
    ax.set_xticklabels([int(year) for year in years], rotation=45)
    
    # Plot unemployment rate on secondary y-axis
    ax2 = ax.twinx()
    ax2.plot(cohort_data['BAYR'], cohort_data['UR'], 's-', color='red', markersize=8)
    ax2.set_ylabel('Unemployment Rate', color='red', fontsize=12)
    ax2.tick_params(axis='y', labelcolor='red')
    
    # Add fitted quadratic trend for field switching
    X = sm.add_constant(np.column_stack((
        cohort_data['grad_year_centered'],
        cohort_data['grad_year_squared']
    )))
    model = sm.OLS(cohort_data['field_switch'], X).fit()
    cohort_data['trend'] = model.predict(X)
    ax.plot(cohort_data['BAYR'], cohort_data['trend'], '--', color='blue', alpha=0.6)
    
    # Add legend
    ax.plot([], [], 's-', color='red', label='Unemployment Rate')
    ax.plot([], [], 'o-', color='blue', label='Field Switching Rate')
    ax.plot([], [], '--', color='blue', label='Fitted Cohort Trend')
    ax.legend(loc='upper right')
    
    plt.tight_layout()
    
    return fig

In [ ]:
fig = create_time_trends_plot(all_masters)


# Time trend figures: by field

In [ ]:
def plot_field_transition_by_field(all_masters, field_names):
    """
    Create separate plots for each field showing the time trends 
    in field switching and unemployment
    """
    import matplotlib.pyplot as plt
    import numpy as np
    import statsmodels.api as sm
    
    # Create a figure for each field
    for field_code, field_name in field_names.items():
        # Filter data for this field
        field_df = all_masters[all_masters['NBAMEMG'] == field_code].copy()
        
        # Skip if we don't have enough data
        if len(field_df) < 100:
            continue
        
        # Create figure
        fig, ax = plt.subplots(figsize=(10, 6))
        
        # Calculate variables needed for trend
        field_df['grad_year_centered'] = field_df['BAYR'] - field_df['BAYR'].min()
        field_df['grad_year_squared'] = field_df['grad_year_centered']**2
        
        # Create aggregate data by cohort
        cohort_data = field_df.groupby('BAYR').agg({
            'field_switch': 'mean',
            'UR': 'mean',
            'grad_year_centered': 'first',
            'grad_year_squared': 'first'
        }).reset_index()
        
        # Plot field switching trend
        ax.plot(cohort_data['BAYR'], cohort_data['field_switch'], 'o-', color='blue', markersize=8)
        ax.set_xlabel('Bachelor\'s Graduation Year', fontsize=12)
        ax.set_ylabel('Field Switching Rate', color='blue', fontsize=12)
        ax.tick_params(axis='y', labelcolor='blue')
        ax.set_title(f'Time Trends for {field_name} Majors', fontsize=14)
        ax.grid(True, linestyle='--', alpha=0.7)
        
        # Set x-axis ticks to integers
        years = cohort_data['BAYR'].unique()
        ax.set_xticks(years)
        ax.set_xticklabels([int(year) for year in years], rotation=45)
        
        # Plot unemployment rate on secondary y-axis
        ax2 = ax.twinx()
        ax2.plot(cohort_data['BAYR'], cohort_data['UR'], 's-', color='red', markersize=8)
        ax2.set_ylabel('Unemployment Rate', color='red', fontsize=12)
        ax2.tick_params(axis='y', labelcolor='red')
        
        # Add fitted quadratic trend for field switching
        X = sm.add_constant(np.column_stack((
            cohort_data['grad_year_centered'],
            cohort_data['grad_year_squared']
        )))
        model = sm.OLS(cohort_data['field_switch'], X).fit()
        cohort_data['trend'] = model.predict(X)
        ax.plot(cohort_data['BAYR'], cohort_data['trend'], '--', color='blue', alpha=0.6)
        
        # Add legend
        ax.plot([], [], 's-', color='red', label='Unemployment Rate')
        ax.plot([], [], 'o-', color='blue', label='Field Switching Rate')
        ax.plot([], [], '--', color='blue', label='Fitted Cohort Trend')
        ax.legend(loc='upper right')
        
#         plt.suptitle(f'Field Switching and Unemployment Relationship for {field_name} Majors', fontsize=16)
        plt.tight_layout()
        
        yield fig, field_name  # Using yield to return each figure separately

# Function to create a single plot for all fields combined
def plot_field_transition_all_fields(all_masters):
    """
    Create a plot for all fields combined showing the time trends
    in field switching and unemployment
    """
    import matplotlib.pyplot as plt
    import numpy as np
    import statsmodels.api as sm
    
    # Create figure
    fig, ax = plt.subplots(figsize=(10, 6))
    
    # Calculate variables needed for trend
    all_masters['grad_year_centered'] = all_masters['BAYR'] - all_masters['BAYR'].min()
    all_masters['grad_year_squared'] = all_masters['grad_year_centered']**2
    
    # Create aggregate data by cohort
    cohort_data = all_masters.groupby('BAYR').agg({
        'field_switch': 'mean',
        'UR': 'mean',
        'grad_year_centered': 'first',
        'grad_year_squared': 'first'
    }).reset_index()
    
    # Plot field switching trend
    ax.plot(cohort_data['BAYR'], cohort_data['field_switch'], 'o-', color='blue', markersize=8)
    ax.set_xlabel('Bachelor\'s Graduation Year', fontsize=12)
    ax.set_ylabel('Field Switching Rate', color='blue', fontsize=12)
    ax.tick_params(axis='y', labelcolor='blue')
    ax.set_title('Time Trends (All Fields Combined)', fontsize=14)
    ax.grid(True, linestyle='--', alpha=0.7)
    
    # Set x-axis ticks to integers
    years = cohort_data['BAYR'].unique()
    ax.set_xticks(years)
    ax.set_xticklabels([int(year) for year in years], rotation=45)
    
    # Plot unemployment rate on secondary y-axis
    ax2 = ax.twinx()
    ax2.plot(cohort_data['BAYR'], cohort_data['UR'], 's-', color='red', markersize=8)
    ax2.set_ylabel('Unemployment Rate', color='red', fontsize=12)
    ax2.tick_params(axis='y', labelcolor='red')
    
    # Add fitted quadratic trend for field switching
    X = sm.add_constant(np.column_stack((
        cohort_data['grad_year_centered'],
        cohort_data['grad_year_squared']
    )))
    model = sm.OLS(cohort_data['field_switch'], X).fit()
    cohort_data['trend'] = model.predict(X)
    ax.plot(cohort_data['BAYR'], cohort_data['trend'], '--', color='blue', alpha=0.6)
    
    # Add legend
    ax.plot([], [], 's-', color='red', label='Unemployment Rate')
    ax.plot([], [], 'o-', color='blue', label='Field Switching Rate')
    ax.plot([], [], '--', color='blue', label='Fitted Cohort Trend')
    ax.legend(loc='upper right')
    
    plt.suptitle('Field Switching and Unemployment Relationship (All Fields Combined)', fontsize=16)
    plt.tight_layout()
    
    return fig

In [ ]:
# After running your prepare_data_minor function:
_, all_masters = prepare_data_major(df_final)

# Define field names
field_names = {
    1: "Computer & Math",
    2: "Bio & Life Sciences",
    3: "Physical Sciences",
    4: "Social Sciences",
    5: "Engineering",
    6: "S&E-Related",
    7: "Non-S&E"
}

# Create field switch indicator based on major groups
all_masters['field_switch'] = (all_masters['NBAMEMG'] != all_masters['NDGMEMG']).astype(int)

# Plot for all fields combined
plot_field_transition_all_fields(all_masters)
# fig_all.savefig('field_switch_all_fields.png')

plot_field_transition_by_field(all_masters, field_names)

# Create individual plots for each field
for fig, field_name in plot_field_transition_by_field(all_masters, field_names):
    # Create a clean filename
    filename = field_name.lower().replace(' ', '_').replace('&', 'and')
    fig.savefig(f'field_switch_{filename}.png')
    plt.close(fig)  # Close the figure to free memory

## Social Science BA

In [ ]:
def analyze_social_science_transitions(all_masters):
    """
    Analyze where Social Science undergraduates go for their master's degrees
    and how these patterns change with economic conditions
    """
    import matplotlib.pyplot as plt
    import statsmodels.api as sm
    import numpy as np
    import pandas as pd
    from scipy.stats import pearsonr
    
    # Define field names for clarity
    field_names = {
        1: "Computer & Math",
        2: "Bio & Life Sciences",
        3: "Physical Sciences",
        4: "Social Sciences",
        5: "Engineering",
        6: "S&E-Related",
        7: "Non-S&E"
    }
    
    # Define a custom color palette with distinct colors
    field_colors = {
        1: '#1f77b4',  # Blue
        2: '#ff7f0e',  # Orange
        3: '#2ca02c',  # Green
        4: '#d62728',  # Red
        5: '#9467bd',  # Purple
        6: '#8c564b',  # Brown
        7: '#7f7f7f',  # Gray (changed from pink)
    }
    
    # Filter for only Social Science undergraduates (NBAMEMG = 4)
    social_sci_grads = all_masters[all_masters['NBAMEMG'] == 4].copy()
    
    # Calculate the distribution of master's fields by cohort
    transitions = social_sci_grads.groupby(['BAYR', 'NDGMEMG']).size().reset_index(name='count')
    
    # Get cohort totals for Social Science grads
    cohort_totals = social_sci_grads.groupby('BAYR').size().reset_index(name='total')
    
    # Calculate transition probabilities
    transitions = transitions.merge(cohort_totals, on='BAYR')
    transitions['probability'] = transitions['count'] / transitions['total']
    
    # Add unemployment rate
    ur_data = social_sci_grads[['BAYR', 'UR']].drop_duplicates().sort_values('BAYR')
    transitions = transitions.merge(ur_data, on='BAYR')
    
    # Calculate correlations between unemployment and transition probabilities for each destination
    destination_correlations = []
    
    for field_code, field_name in field_names.items():
        field_data = transitions[transitions['NDGMEMG'] == field_code]
        
        if len(field_data) >= 5:  # Only calculate if we have enough data points
            corr, p_val = pearsonr(field_data['UR'], field_data['probability'])
            destination_correlations.append({
                'destination_code': field_code,
                'destination_name': field_name,
                'correlation': corr,
                'p_value': p_val
            })
    
    # Create a summary graph showing destination fields for social science majors
    fig1, ax1 = plt.subplots(figsize=(12, 8), facecolor='white')
    ax1.set_facecolor('white')
    
    # Create a pivot table for visualization
    pivot_data = transitions.pivot(index='BAYR', columns='NDGMEMG', values='probability').fillna(0)
    
    # Make sure all field codes are in the pivot data (fill with zeros if not)
    for field_code in field_names.keys():
        if field_code not in pivot_data.columns:
            pivot_data[field_code] = 0
    
    # Sort columns to match field_names order
    existing_columns = [col for col in sorted(pivot_data.columns) if col in field_names]
    pivot_data = pivot_data[existing_columns]
    
    # Create a dictionary mapping column numbers to colors
    color_map = {col: field_colors.get(col, '#333333') for col in pivot_data.columns}
    
    # Plot stacked area chart with custom colors
    area_plot = pivot_data.plot.area(ax=ax1, stacked=True, color=[color_map[col] for col in pivot_data.columns])
    
    # Add labels and title
    ax1.set_xlabel('Bachelor\'s Graduation Year')
    ax1.set_ylabel('Proportion of Social Science Graduates')
    ax1.set_title('Destination Fields for Social Science Undergraduates')
    
    # Format x-axis as integers
    ax1.set_xticks(pivot_data.index)
    ax1.set_xticklabels([int(year) for year in pivot_data.index], rotation=45)
    
    # Add unemployment rate as a dashed black line on secondary y-axis
    ax1_twin = ax1.twinx()
    ax1_twin.plot(ur_data['BAYR'], ur_data['UR'], 'k--', linewidth=2, label='Unemployment Rate')
    ax1_twin.set_ylabel('Unemployment Rate (%)', color='black')
    ax1_twin.tick_params(axis='y', labelcolor='black')
    
    # Fix the legend
    # Remove the auto-generated legend
    ax1.get_legend().remove()
    
    # Create custom legend handles and labels
    legend_handles = []
    legend_labels = []
    
    # Create patches for each field in the correct order (bottom to top in the stack)
    for field_code in pivot_data.columns:
        color = color_map[field_code]
        patch = plt.Rectangle((0,0), 1, 1, fc=color)
        legend_handles.append(patch)
        legend_labels.append(field_names[field_code])
    
    # Add unemployment rate to legend
    ur_line = plt.Line2D([0], [0], color='black', linestyle='--', linewidth=2)
    legend_handles.append(ur_line)
    legend_labels.append('Unemployment Rate')
    
    # Add the custom legend in reverse order (to match bottom-to-top stacking)
    # But keep unemployment rate at the end
    field_handles = legend_handles[:-1]
    field_labels = legend_labels[:-1]
    ax1.legend(field_handles[::-1] + [legend_handles[-1]], 
              field_labels[::-1] + [legend_labels[-1]], 
              loc='upper right', title='Master\'s Field')
    
    # Rest of the code remains the same...
    
    # Create a third figure showing time series for selected destinations
    fig3, ax3 = plt.subplots(figsize=(14, 8))
    
    # Plot unemployment rate on secondary y-axis
    ax4 = ax3.twinx()
    ax4.plot(ur_data['BAYR'], ur_data['UR'], 's-', color='red', linewidth=2, label='Unemployment Rate')
    ax4.set_ylabel('Unemployment Rate', color='red')
    ax4.tick_params(axis='y', labelcolor='red')
    
    # Plot each destination field's probability over time
    for field_code, field_name in field_names.items():
        field_data = transitions[transitions['NDGMEMG'] == field_code]
        if len(field_data) >= 5:
            ax3.plot(field_data['BAYR'], field_data['probability'], 'o-', 
                    color=field_colors.get(field_code, '#333333'), 
                    linewidth=2, 
                    label=field_name)
    
    # Add labels and title
    ax3.set_xlabel('Bachelor\'s Graduation Year')
    ax3.set_ylabel('Transition Probability')
    ax3.set_title('Where Social Science Undergraduates Go for Master\'s Degrees')
    
    # Format x-axis as integers
    ax3.set_xticks(ur_data['BAYR'])
    ax3.set_xticklabels([int(year) for year in ur_data['BAYR']], rotation=45)
    
    # Add grid and legend
    ax3.grid(True, alpha=0.3)
    
    # Create combined legend
    lines1, labels1 = ax3.get_legend_handles_labels()
    lines2, labels2 = ax4.get_legend_handles_labels()
    ax3.legend(lines1 + lines2, labels1 + labels2, loc='upper left', bbox_to_anchor=(1, 1))
    
    plt.tight_layout()
    
    return {
        'stacked_area': fig1,
        'time_series': fig3,
        'correlations': dest_corr_df if 'dest_corr_df' in locals() else None
    }

In [ ]:
analyze_social_science_transitions(all_masters)